# Exp 031 — CoT user-state prompt + Qwen 2.5-3B (DEVSET)

**Three-way pair-test sibling of 029 / 030.** Same retrieval (wRRF), same CoT prompt (`response_generation_cot_user_state.txt`), same `max_new_tokens=192`. Only `lm_type` changes:

- 029: Qwen 2.5-**1.5B** (champion model, ~67% format compliance in local smoke)
- **031: Qwen 2.5-3B** (this notebook — practical sweet spot, expected ~85%)
- 030: Qwen 2.5-**7B** (slow at batch 4 due to OOM)

**Why 3B is the practical sweet spot**: large enough to follow the structured `<user_state>` block reliably (the confound that hit 1.5B), small enough to run 8000 rows in ~12-18 min at batch 16 (vs 100+ min for 7B at batch 4). And the CoT prompt's word-bans target exactly the AI-speak failure mode that hit 3B + stock prompt on Blind-A (exp 024).

**Risk being tested**: exp 024 shipped Qwen 3B + stock prompt + max_new_tokens=192 to Blind-A and it regressed (composite 0.33→0.29, LLM 3.15→3.00). The new CoT prompt explicitly bans 'fantastic', 'perfectly captures', etc. If 3B+CoT cleanly produces grounded responses on dev, the 3B path opens; if responses still drift to filler, the bigger-model penalty is structural and we revert to 1.5B.

Wall time on A100: ~12-18 min for 8000 rows at batch 16.

In [ ]:
# 1) Verify GPU. 3B fits T4/L4/A100; A100 strongly recommended for batch 16 speed.
!nvidia-smi | head -20

In [ ]:
# 2) FORCE-FRESH clone — pull latest fresh-model code.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026-lora-tutorial
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026-lora-tutorial
%cd /content/recsys2026-lora-tutorial
print('\n=== CODE VERSION CHECK ===')
!git log -1 --pretty=format:'commit:  %h%ndate:    %ai%nsubject: %s'
print()
!echo -n 'branch:  ' && git rev-parse --abbrev-ref HEAD

In [ ]:
# 2b) Mount Drive + wire persistent caches + persistent Python packages.
# Persists across Colab sessions:
#   * HF datasets (talkpl-ai/* — challenge data)
#   * experiments/cache (BM25/dense/cf-bpr indices)
#   * Python packages installed via pip --target=PERSISTENT_PKG (e.g. vllm)
#
# Drive auth: a popup appears the first time. CLICK THROUGH ALL
# permission screens — closing the popup early causes
# 'credential propagation was unsuccessful'. If you hit that error,
# this cell will retry with force_remount=True; if THAT also fails,
# you may need to clear cookies for accounts.google.com or use an
# incognito window.
import os, sys, shutil
from google.colab import drive

try:
    drive.mount('/content/drive')
except Exception as e:
    print(f'first mount attempt failed: {e}\nretrying with force_remount=True ...')
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    drive.mount('/content/drive', force_remount=True)

# Sanity check the mount is actually live before proceeding.
assert os.path.isdir('/content/drive/MyDrive'), (
    'Drive mount failed — /content/drive/MyDrive does not exist. '
    'Common fixes: (1) complete the OAuth popup fully — every permission screen, '
    "don't close mid-flow, (2) disable popup blocker for colab.research.google.com, "
    '(3) sign in to Google in this browser tab on the SAME account before re-running.'
)

DRIVE_BASE = '/content/drive/MyDrive/recsys2026-cache'
PERSISTENT_PKG = '/content/drive/MyDrive/python_packages'
for d in [f'{DRIVE_BASE}/hf_datasets', f'{DRIVE_BASE}/experiments_cache', PERSISTENT_PKG]:
    os.makedirs(d, exist_ok=True)

os.environ['HF_DATASETS_CACHE'] = f'{DRIVE_BASE}/hf_datasets'
%env HF_DATASETS_CACHE={DRIVE_BASE}/hf_datasets

if PERSISTENT_PKG not in sys.path:
    sys.path.insert(0, PERSISTENT_PKG)
%env PYTHONPATH={PERSISTENT_PKG}

EXPECTED_CACHE = '/content/recsys2026-lora-tutorial/experiments/cache'
os.makedirs(os.path.dirname(EXPECTED_CACHE), exist_ok=True)
if os.path.exists(EXPECTED_CACHE) and not os.path.islink(EXPECTED_CACHE):
    shutil.rmtree(EXPECTED_CACHE)
if not os.path.islink(EXPECTED_CACHE):
    os.symlink(f'{DRIVE_BASE}/experiments_cache', EXPECTED_CACHE)

print(f'datasets cache  : {os.environ["HF_DATASETS_CACHE"]}')
print(f'experiments dir : {EXPECTED_CACHE} -> {os.readlink(EXPECTED_CACHE)}')
print(f'python packages : {PERSISTENT_PKG}')
!ls -lh {DRIVE_BASE}/
!ls {PERSISTENT_PKG}/ 2>/dev/null | head -10 || echo '  (empty — first run)'

In [ ]:
# 3) Install deps + vLLM (vLLM persisted to Drive via cell 2b).
#
# vLLM install strategy: --no-deps to SKIP pip's dependency resolver
# (which spends 5-10 min backtracking through ~30 conflict warnings
# from Colab's pre-installed gradio/cuml/tensorflow/etc. stack — most
# of those are irrelevant to us). vLLM bundles its own CUDA kernels;
# the Python deps it actually imports (torch, transformers, ...) are
# already in Colab's base image at compatible versions.
#
# If --no-deps install fails at import time because some specific dep
# really IS missing, fall through to a full install (slower but
# resolves the missing piece).
!pip install -q -r requirements.txt

import importlib
importlib.invalidate_caches()
try:
    import vllm  # noqa: F401
    print(f'vllm {vllm.__version__} already on Drive ✓ (skipped install)')
except ImportError:
    print('vllm not on Drive — installing to persistent dir (no-deps fast path)...')
    !pip install -q --no-deps --target={PERSISTENT_PKG} vllm
    importlib.invalidate_caches()
    try:
        import vllm
        print(f'installed (no-deps): vllm {vllm.__version__}')
    except ImportError as e:
        # Some genuine dep is missing. Re-run with full deps (slower).
        print(f'no-deps import failed ({e}); falling back to full install (slow)...')
        !pip install -q --target={PERSISTENT_PKG} vllm
        importlib.invalidate_caches()
        import vllm
        print(f'installed (full deps): vllm {vllm.__version__}')

!python -c "import torch, transformers, bm25s; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"
# Verify the subprocess can see vllm too (PYTHONPATH set in cell 2b).
!python -c "import vllm; print('subprocess sees vllm:', vllm.__version__)"

In [ ]:
# 4) Experiment parameters.
TID = '031-cot-user-state-qwen3b-devset'
# vLLM is configured via use_vllm: true in the yaml. It manages its own
# memory pool (gpu_memory_utilization=0.9 = pre-allocates ~36 GB on the
# A100 40GB as a shared KV cache, NOT per-batch). So BATCH_SIZE here is
# really a "cap on how many sequences we hand vLLM at once" — vLLM will
# continuous-batch them internally regardless. Keeping it at a high value
# (256) gives vLLM plenty of in-flight sequences to schedule.
BATCH_SIZE = 256
# ATTN is ignored under vLLM — it picks Flash Attention 2 internally
# on Ampere+ GPUs. Kept here for symmetry with sdpa fallback path.
ATTN = 'flash_attention_2'
print(f'BATCH_SIZE={BATCH_SIZE}  ATTN(ignored under vLLM)={ATTN}')
# import os; os.environ['HF_TOKEN'] = 'hf_...'

In [ ]:
# 5) Run devset two-step inference.
#
# Env vars threaded into the subprocess:
#   PYTHONPATH=PERSISTENT_PKG                    -> sees vllm from Drive
#   PYTORCH_ALLOC_CONF=expandable_segments:True  -> reduces fragmentation
#   VLLM_USE_V1=0                                -> in-process engine
#       (avoids EngineCore subprocess CUDA init failure on Colab)
#   VLLM_WORKER_MULTIPROC_METHOD=spawn           -> avoid fork CUDA breakage
!cd music-crs-baselines && \
    PYTHONPATH={PERSISTENT_PKG} \
    PYTORCH_ALLOC_CONF=expandable_segments:True \
    VLLM_USE_V1=0 \
    VLLM_WORKER_MULTIPROC_METHOD=spawn \
    python run_inference_devset.py \
    --tid {TID} \
    --batch_size {BATCH_SIZE} \
    --device cuda \
    --attn_implementation {ATTN}

In [ ]:
# 6) Validate prediction JSON + zip for download.
import json, os, shutil
SRC = f'music-crs-baselines/exp/inference/devset/{TID}.json'
assert os.path.isfile(SRC), f'prediction not found at {SRC} — did inference fail?'

with open(SRC) as f:
    rows = json.load(f)
print(f'rows: {len(rows)} (expected 8000)')
assert len(rows) >= 8000, f'only {len(rows)} rows — partial run; do not score'

stage = f'/content/_stage_{TID}'
shutil.rmtree(stage, ignore_errors=True)
os.makedirs(stage, exist_ok=True)
shutil.copy(SRC, os.path.join(stage, f'{TID}.json'))
zip_base = f'/content/{TID}'
shutil.make_archive(zip_base, 'zip', stage)
print('wrote', zip_base + '.zip')
!ls -lh {zip_base}.zip

In [ ]:
# 7a) Browser download.
from google.colab import files
files.download(f'/content/{TID}.zip')

In [ ]:
# 7b) Drive backup.
from google.colab import drive
import os, shutil
drive.mount('/content/drive')
dst_dir = '/content/drive/MyDrive/recsys2026-predictions'
os.makedirs(dst_dir, exist_ok=True)
shutil.copy(f'/content/{TID}.zip', dst_dir)
shutil.copy(f'music-crs-baselines/exp/inference/devset/{TID}.json', dst_dir)
print(f'saved to Drive: {dst_dir}')
!ls -lh {dst_dir}

In [ ]:
# 8) Quality probe — sample responses + parser-leak rate.
# The CoT post-processor in crs_baseline.extract_cot_response prefers
# <response>...</response>; if missing, it strips <user_state>...</user_state>
# and returns the rest. The FINAL parsed text going to Gemini should be
# near-zero leak (no <user_state> / <response> tags, no field names like 'mood:').
import json, random, re

with open(f'music-crs-baselines/exp/inference/devset/{TID}.json') as f:
    rows = json.load(f)

field_leak_re = re.compile(
    r'^(?:mood|intent|energy|sonic_pref|era_pref|familiarity):',
    re.M,
)
tag_leak_re = re.compile(r'<\s*/?\s*(user_state|response)\s*>', re.I)

leak_field, leak_tag, empty = 0, 0, 0
for r in rows:
    resp = (r.get('predicted_response') or '').strip()
    if not resp:
        empty += 1
        continue
    if field_leak_re.search(resp):
        leak_field += 1
    if tag_leak_re.search(resp):
        leak_tag += 1

n = len(rows)
print(f'rows total            : {n}')
print(f'empty responses       : {empty}  ({empty/n:.1%})')
print(f'field-name leak       : {leak_field}  ({leak_field/n:.1%})')
print(f'tag leak              : {leak_tag}  ({leak_tag/n:.1%})')
print()
print('=== 10 random sample responses ===')
random.seed(42)
for i in random.sample(range(n), min(10, n)):
    print(f'\n[{i}] turn {rows[i].get("turn_number")}')
    print(rows[i].get('predicted_response', '')[:400])

# Heuristic gate: tag/field-leak rate >5% in the FINAL parsed response
# means parser failure (unexpected — local smoke had 0% leak). It is fine
# if the LM only emits the structured <user_state> block ~70% of the time
# (1.5B drops the format on some queries — local smoke showed ~67%
# follow-rate). The parser falls back cleanly when the format is missing.

## After the Colab run, on local M4:

```bash
cd recsys2026
TID=031-cot-user-state-qwen3b-devset
unzip -o ~/Downloads/${TID}.zip -d music-crs-baselines/exp/inference/devset/
source recsys26/bin/activate
python scripts/local_eval.py --tid ${TID} --split dev
pytest tests/test_wave2_integration.py -v
```

## Decision gate (compared to 029 / 1.5B-CoT and exp 024 / 3B-stock-prompt)

- **Response quality clearly more grounded than 029 AND no AI-speak filler ('fantastic', 'perfectly')** → 3B+CoT is the new candidate; promote to a Blind-A run (still subject to fresh-model gate).
- **Format-compliance ≥85% AND comparable response specificity to 029** → CoT is the dominant lever; 1.5B cheaper and equally good for production.
- **Filler words leak through despite the prompt's bans** → 3B's stock-prompt failure mode (exp 024) is structural; revert to 1.5B.